# 점검 2 — 지점별 누적 관측시간

**목적**: 송도 20개 교차로 각각이 실제로 몇 시간 촬영됐는지 확인한다. 이 숫자로 (1) 지점별로 EVT를 따로 돌릴 수 있는지, 아니면 구조(4지/3지)별로 합쳐야 하는지 결정하고, (2) 논문 Data 절에 "지점당 관측시간"으로 정직하게 적는다.

**비교 기준(메인 논문)**
- Zheng & Sayed (2019, TR-C): 교차로 2곳, 지점당 15~17시간
- Zheng·Sayed·Essa (2019, AAP): 교차로 4곳, 지점당 1~2시간 → 논문 스스로 "관측기간이 짧아 신뢰구간이 넓다"고 한계로 밝힘

**입력**: `data/raw/{날짜}_{교차로}/{날짜}_{교차로}_{세션}.csv` (지점당 4일 × 10세션 = 40파일, 총 800파일)

**쓰는 컬럼**: `Local_Time`(HH:MM:SS.mmm, 날짜는 파일명에 있음), `Drone_ID`(같은 교차로를 드론 2대가 찍은 경우 중복 방지), `Vehicle_ID`(참고용 대수)

**방법**: 파일마다 궤적점이 1개 이상 있는 '초'를 세서 합산한다. 드론이 세션 도중 옆 교차로로 넘어가 있던 시간은 궤적점이 없으므로 자동으로 빠진다.

**원본 데이터는 읽기만 하고 수정하지 않는다.** 결과표는 `analysis/outputs/`에 저장한다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path('C:/Users/123/Documents/(송도) 교통 연구 논문/data/raw')
INTERSECTIONS = list('ABCEFGHIJKLMNOPQRSTU')   # D 제외 20곳 (D는 Drone 약자)
DAYS = ['2022-10-04', '2022-10-05', '2022-10-06', '2022-10-07']
SESSIONS = ['AM1', 'AM2', 'AM3', 'AM4', 'AM5', 'PM1', 'PM2', 'PM3', 'PM4', 'PM5']
FOUR_LEG  = list('ABFHIJKLMNOPRST')   # 4지(사거리) 15곳 — segmentations node 수로 확인
THREE_LEG = list('CEGQU')             # 3지(T자) 5곳

print('교차로 수:', len(INTERSECTIONS), '| 4지:', len(FOUR_LEG), '| 3지:', len(THREE_LEG))
print('데이터 폴더 존재:', DATA_DIR.exists())

## 1. 파일 하나를 요약하는 함수

한 파일 = 교차로 1곳의 세션 1개(약 30분 슬롯). 여기서 세 가지를 뽑는다.

| 값 | 뜻 |
|---|---|
| `span_min` | 첫 궤적점부터 마지막 궤적점까지의 시간폭 (드론이 자리를 비운 시간 포함) |
| `covered_min` | 궤적점이 1개 이상 있는 '초'의 개수 ÷ 60 = **실제로 찍혀 있던 시간** |
| `gap_min` | span − covered = 드론이 이 교차로를 안 찍고 있던 시간 |

`covered`가 우리가 논문에 쓸 숫자다. 차량이 한 대도 없는 초는 안 세지므로 아주 약간 과소평가될 수 있지만, 송도 교차로 교통량에서는 무시할 수준이다.

In [ ]:
def summarize_file(csv_path: Path) -> dict:
    """CSV 파일 하나(교차로 1곳 × 세션 1개)의 관측시간 요약."""
    df = pd.read_csv(csv_path, usecols=['Vehicle_ID', 'Local_Time', 'Drone_ID'],
                     dtype={'Local_Time': str})
    if len(df) == 0:
        return {'file': csv_path.name, 'n_rows': 0}

    # 'HH:MM:SS.mmm' 문자열 -> 그날 0시 기준 초(float)
    sec = pd.to_timedelta(df['Local_Time']).dt.total_seconds().to_numpy()
    sec_bin = np.floor(sec).astype(np.int64)          # 1초 단위로 내림

    covered_s = int(np.unique(sec_bin).size)          # 궤적점이 있는 초의 개수 (드론 중복은 1회만)
    span_s = float(sec.max() - sec.min())

    drones = sorted(df['Drone_ID'].unique().tolist())
    drone_cov = {int(d): int(np.unique(sec_bin[df['Drone_ID'].to_numpy() == d]).size) for d in drones}

    day, inter, sess = csv_path.stem.split('_')       # 예: 2022-10-04_A_AM1
    return {
        'file': csv_path.name, 'day': day, 'intersection': inter, 'session': sess,
        'n_rows': int(len(df)), 'n_vehicles': int(df['Vehicle_ID'].nunique()),
        'drones': ','.join(str(d) for d in drones),
        'drone_cov_s': str(drone_cov),
        't_first': df['Local_Time'].min(), 't_last': df['Local_Time'].max(),
        'span_min': round(span_s / 60, 2),
        'covered_min': round(covered_s / 60, 2),
        'gap_min': round((span_s - covered_s) / 60, 2),
    }

## 2. 파일 하나로 먼저 확인

800개를 돌리기 전에 A 교차로 첫날 첫 세션 하나로 값이 말이 되는지 본다.

확인 포인트: `t_first`~`t_last`가 아침 7시대인지, `span_min`이 30분 안팎인지, `covered_min`이 그보다 작은지(드론이 옆 교차로로 갔다 온 만큼), `drones`에 드론 번호가 몇 개 나오는지.

In [ ]:
test_file = DATA_DIR / '2022-10-04_A' / '2022-10-04_A_AM1.csv'
r = summarize_file(test_file)
for k, v in r.items():
    print(f'{k:>12}: {v}')

## 3. 첫날(20곳 × 10세션 = 200파일)만 먼저 스캔

파일마다 약 50MB라 하루치도 몇 분 걸린다. 진행 상황이 날짜별로 출력된다. 폴더나 파일이 없으면 이름을 찍어주니 누락이 있는지 여기서 잡는다.

In [ ]:
def scan(days):
    rows = []
    for day in days:
        for x in INTERSECTIONS:
            folder = DATA_DIR / f'{day}_{x}'
            if not folder.exists():
                print('폴더 없음:', folder.name)
                continue
            for s in SESSIONS:
                f = folder / f'{day}_{x}_{s}.csv'
                if not f.exists():
                    print('파일 없음:', f.name)
                    continue
                rows.append(summarize_file(f))
        print(day, '완료 | 누적 파일 수:', len(rows))
    return pd.DataFrame(rows)

res_day1 = scan(DAYS[:1])
res_day1.head(10)

## 4. 4일 전체 스캔 + 저장

800파일 전체. 수십 분 걸릴 수 있으니 돌려놓고 기다린다. 끝나면 `analysis/outputs/01_관측시간_파일별.csv`로 저장된다(원본은 건드리지 않음).

In [ ]:
res = scan(DAYS)

OUT_DIR = Path.cwd() / 'outputs'
OUT_DIR.mkdir(exist_ok=True)
out_path = OUT_DIR / '01_관측시간_파일별.csv'
res.to_csv(out_path, index=False, encoding='utf-8-sig')
print('저장:', out_path, '| 파일 수:', len(res))

## 5. 지점별 집계

여기서 나오는 `covered_h`가 논문에 쓸 "지점당 누적 관측시간"이다.

확인 포인트
- `files`가 40이 아닌 지점이 있으면 파일 누락
- `covered_h`와 `span_h` 차이가 크면 그 지점은 드론이 자주 자리를 비웠다는 뜻
- `n_vehicles` 합은 궤적 분절 때문에 실제 차량 수보다 부풀려질 수 있음(원논문 Appendix D). 참고용으로만.

In [ ]:
per_site = (res.groupby('intersection')
              .agg(files=('file', 'count'),
                   covered_h=('covered_min', lambda s: round(s.sum() / 60, 2)),
                   span_h=('span_min', lambda s: round(s.sum() / 60, 2)),
                   vehicles=('n_vehicles', 'sum'),
                   rows=('n_rows', 'sum'))
              .reset_index())
per_site['structure'] = np.where(per_site['intersection'].isin(FOUR_LEG), '4지', '3지')
per_site = per_site.sort_values('covered_h', ascending=False).reset_index(drop=True)
per_site

In [ ]:
# 지점 × 날짜 누적 관측시간 (시간 단위)
by_day = res.pivot_table(index='intersection', columns='day', values='covered_min', aggfunc='sum').div(60).round(2)
by_day['total_h'] = by_day.sum(axis=1).round(2)
display(by_day)

# 구조별 합계
print(per_site.groupby('structure')[['covered_h', 'vehicles']].sum())

# 선행연구 기준선과 비교
print('\n비교: Zheng & Sayed 2019 = 지점당 15~17h (2곳) | Zheng·Sayed·Essa 2019 = 지점당 1~2h (4곳)')
print('송도 지점당 누적 관측시간 min / median / max (h):',
      per_site['covered_h'].min(), per_site['covered_h'].median(), per_site['covered_h'].max())

## 6. 결과를 어떻게 읽을 것인가

- **지점당 covered_h가 10시간 안팎 이상**: 지점별 단변량 EVT는 가능. 이변량(joint exceedance가 필요)은 지점에 따라 얇을 수 있으니 점검 1 결과와 함께 판단.
- **지점당 covered_h가 5시간 미만**: 지점별 이변량은 어렵다고 보고, 구조별(4지 15곳 / 3지 5곳) 합산 설계를 주축으로 잡는다.
- 어느 쪽이든 논문 Data 절에는 "20곳, 지점당 X~Y시간, 총 Z시간"으로 그대로 적는다. 선행연구(2곳 15~17h / 4곳 1~2h)와 나란히 놓으면 "지점 수는 많고 지점당 시간은 짧거나 비슷하다"는 정직한 비교가 된다.

다음 노트북(점검 1)은 4지 교차로 한 곳에서 좌회전 궤적과 맞은편 직진 궤적이 교차로 안에 동시에 있는지를 센다. 그 결과로 상충유형(crossing vs 후미추돌)을 정한다.